# Assignment 2: Diabetes Classification

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('diabetes_binary_classification_data.csv')

# All values are whole numbers stored as floats, so we store them as integers
df = df.astype(int)

print(df.shape)
df.head()

(253680, 22)


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0,1,1,1,40,1,0,0,0,0,...,1,0,5,18,15,1,0,9,4,3
1,0,0,0,0,25,1,0,0,1,0,...,0,1,3,0,0,0,0,7,6,1
2,0,1,1,1,28,0,0,0,0,1,...,1,1,5,30,30,1,0,9,4,8
3,0,1,0,1,27,0,0,0,1,1,...,1,0,2,0,0,0,0,11,3,6
4,0,1,1,1,24,0,0,0,1,1,...,1,0,2,3,0,0,0,11,5,4


# Step 1 - Understand the data

## Features and target

- **Target (y):** `Diabetes_binary`
- **Input features (X):** the other 21 columns

In [2]:
X = df.drop(columns=['Diabetes_binary'])
y = df['Diabetes_binary']

print('X:', X.shape)
print('y:', y.shape)
print('Missing values:', df.isna().sum().sum())

X: (253680, 21)
y: (253680,)
Missing values: 0


## What do class 0 and class 1 mean?

- **0** = no diabetes
- **1** = prediabetes or diabetes

Class 1 combines prediabetes and diabetes, so the model cannot tell them apart.

## How many observations are in each class?

In [3]:
class_counts = pd.DataFrame({
    'Count': y.value_counts(),
    'Proportion': y.value_counts(normalize=True).round(3),
})
class_counts

,Count,Proportion
Diabetes_binary,,
0,218334,0.861
1,35346,0.139


The target is **imbalanced**: only about 14% are class 1. A model that always predicts "no diabetes" would already be about 86% accurate, so accuracy alone is not enough to judge the model. We also need precision and recall.

## Three features of different types

All columns are stored as numbers, but the numbers mean different things. We use the *Understanding the Dataset* guide to read them:

| Feature | Type | What the values mean |
|---|---|---|
| `BMI` | Numerical | Body Mass Index, a real measurement (12-98) |
| `HighBP` | Binary | 0 = no high blood pressure, 1 = high blood pressure |
| `GenHlth` | Ordered | Self-rated health: 1 = excellent, 2 = very good, 3 = good, 4 = fair, 5 = poor |

In [4]:
print(df['BMI'].describe().round(1), '\n')
print(df['HighBP'].value_counts().sort_index(), '\n')
print(df['GenHlth'].value_counts().sort_index())

count    253680.0
mean         28.4
std           6.6
min          12.0
25%          24.0
50%          27.0
75%          31.0
max          98.0
Name: BMI, dtype: float64 

HighBP
0    144851
1    108829
Name: count, dtype: int64 

GenHlth
1    45299
2    89084
3    75646
4    31570
5    12081
Name: count, dtype: int64


- `BMI` is a real amount, so a higher number means a higher BMI.
- `HighBP` is a yes/no answer, so 0 and 1 are two categories, not amounts.
- `GenHlth` has an order (higher = worse health), but the steps between codes are not measured distances. A 4 is not "twice as bad" as a 2.

So looking only at the numbers in the CSV is not enough. The same kind of number can be a measurement, a category or an ordered group.